# Phishing Email Detection with an SVM
A baseline phishing/spam email classifier, a check for dataset shortcuts, and a test on real samples.

**How to run (Google Colab):** upload `phishing_email.csv` through the folder panel on the left, wait for the upload to finish, then click **Runtime > Run all**. The dataset is not included in this repository (see the README for where to get it).

## 1. Load and look at the data
The file has two columns: `text_combined` (the email text, already lowercased and stripped of punctuation) and `label` (0 = legitimate, 1 = phishing or spam).

In [ ]:
import os
import pandas as pd

DATA_PATH = next(p for p in ["/content/phishing_email.csv", "/phishing_email.csv", "phishing_email.csv"] if os.path.exists(p))
df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.columns)
print(df.head())

for c in df.columns:
    if df[c].nunique() <= 5:
        print(df[c].value_counts())

## 2. Clean the data
Remove blank rows and duplicate emails. Duplicates matter because the same email could land in both the training and test sets, which would flatter the score.

In [ ]:
print(df.isna().sum())
print("duplicates:", df.duplicated().sum())

df = df.dropna().drop_duplicates()
df["label"] = df["label"].astype(int)
print(df.shape)
print(df["label"].value_counts())

## 3. Baseline: TF-IDF + linear SVM
80% of the emails train the model and 20% are held back as a test set it never sees. TF-IDF turns each email into numbers (up to 20,000 words and two-word phrases), and a linear SVM learns the dividing line between the two classes.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    df["text_combined"], df["label"],
    test_size=0.2, random_state=42, stratify=df["label"]
)

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

model = LinearSVC()
model.fit(X_train_vec, y_train)

pred = model.predict(X_test_vec)
print(classification_report(y_test, pred, target_names=["legit", "phishing"]))
print(confusion_matrix(y_test, pred))

## 4. What did the model learn?
The words with the highest and lowest weights show what the model relies on. If they are mostly about where an email came from (years, names), the model is recognising the source and not only the scam.

In [ ]:
import numpy as np

words = np.array(tfidf.get_feature_names_out())
weights = model.coef_[0]

print("Top clues for PHISHING:")
print(words[np.argsort(weights)[-20:]][::-1])

print("\nTop clues for LEGIT:")
print(words[np.argsort(weights)[:20]])

## 5. Error analysis: which phishing emails slipped through?

In [ ]:
test_df = X_test.reset_index(drop=True).to_frame()
test_df["true"] = y_test.reset_index(drop=True)
test_df["pred"] = pred

missed = test_df[(test_df["true"] == 1) & (test_df["pred"] == 0)]
print(len(missed))
for t in missed["text_combined"].head(5):
    print(t[:300], "\n---")

## 6. Shortcut check
Remove all numbers (years, area codes, time codes) and ban the source-specific names spotted in section 4, then retrain and compare.

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def strip_numbers(t):
    return re.sub(r"\d+", " ", t)

X_train_c = X_train.apply(strip_numbers)
X_test_c = X_test.apply(strip_numbers)

leaky = ["enron", "vince", "louise", "daren", "josemonkeyorg",
         "opensuse", "uai", "ierant", "fork", "edu"]

tfidf2 = TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                         stop_words=list(ENGLISH_STOP_WORDS) + leaky)
Xtr = tfidf2.fit_transform(X_train_c)
Xte = tfidf2.transform(X_test_c)

model2 = LinearSVC()
model2.fit(Xtr, y_train)
pred2 = model2.predict(Xte)

print(classification_report(y_test, pred2, target_names=["legit", "phishing"]))
print(confusion_matrix(y_test, pred2))

## 7. Hand-written test emails
Three short emails written for this test: a classic scam, a normal work email, and a polite business-style lure that imitates the kind of email that slipped through in section 5.

In [ ]:
tests = {
    "business-style lure": "dear jose please find the attached pdf kind attention open the pdf thanks",
    "classic scam": "your account has been suspended verify your password now click here immediately",
    "normal work email": "hi team please see the attached schedule for next week meeting thanks",
}

for name, text in tests.items():
    t = strip_numbers(text.lower())
    p1 = model.predict(tfidf.transform([t]))[0]
    p2 = model2.predict(tfidf2.transform([t]))[0]
    print(name, "| model 1:", "phishing" if p1 else "legit", "| model 2:", "phishing" if p2 else "legit")

## 8. Test on two real phishing emails
Two phishing emails from the sample set analysed in my phishing forensics project, typed in by hand as lowercase body text (no headers, addresses or attachment data). The score shows how far the model leans: above 0 means phishing, below 0 means legit.

In [ ]:
real_emails = {
    "black friday voucher": """black friday sale 50% discount on all shops starting now until wednesday the 31th of decembr we will be offering all users a 45% on discount any in online shop this means that if you haven't upgraded yet here is your chance use the coupon for any purchases in online shop amazon and other online shops coupon attached""",
    "invoice with zip": """dear all we've made balance payment for attached invoice on 14/12/2017 our below forwarder will contact your side for pickup arrangement evo logistics pte ltd no 7 airline road cargo agent building e singapore pic lucy tiew there's no need to send the original tax invoice or declaration letter together with the goods thank you huong le""",
}

for name, text in real_emails.items():
    t = strip_numbers(text.lower())
    s1 = model.decision_function(tfidf.transform([t]))[0]
    s2 = model2.decision_function(tfidf2.transform([t]))[0]
    print(name)
    print("  model 1:", "phishing" if s1 > 0 else "legit", "| score", round(s1, 2))
    print("  model 2:", "phishing" if s2 > 0 else "legit", "| score", round(s2, 2))